# MeLi Data Challenge 2021 — Análisis Exploratorio
Dataset de ventas de MercadoLibre para predecir días hasta agotar stock.
Archivos: historial de ventas (parquet), datos de test (csv), metadata de items (jl).

In [36]:
import pandas as pd

In [37]:
rut_parquet = "data/meli_train_data.parquet"
rut_csv = "data/meli_test_data.csv"
rut_jl = "data/items_static_metadata.jl"

In [38]:
archivo_parquet = pd.read_parquet(rut_parquet)
archivo_csv = pd.read_csv(rut_csv)
archivo_jl = pd.read_json(rut_jl, lines = True)

## Estructura de los archivos
Vista previa de los tres datasets para entender columnas y formato.

In [39]:
archivo_jl.head()

,item_domain_id,item_id,item_title,site_id,sku,product_id,product_family_id
0,MLB-SNEAKERS,492155,Tênis Masculino Olympikus Cyber Barato Promoçao,MLB,0,NaN,MLB15832732
1,MLB-SURFBOARD_RACKS,300279,Suporte Rack Prancha Parede C/ Regulagem Horiz...,MLB,1,NaN,NaN
2,MLM-NECKLACES,69847,5 Collares Plateados Dama Gargantilla Choker -...,MLM,2,NaN,NaN
3,MLM-RINGS,298603,Lindo Anillo De Bella Crepusculo Twilight Prom...,MLM,3,NaN,NaN
4,MLB-WEBCAMS,345949,Webcam Com Microfone Hd 720p Knup Youtube Pc V...,MLB,4,NaN,NaN


In [40]:
archivo_csv.head()

,sku,target_stock
0,464801,3
1,645793,4
2,99516,8
3,538100,8
4,557191,10


In [41]:
archivo_parquet.head()

,sku,date,sold_quantity,current_price,currency,listing_type,shipping_logistic_type,shipping_payment,minutes_active
0,464801,2021-02-01,0,156.78,REA,classic,fulfillment,free_shipping,1440.0
1,464801,2021-02-02,0,156.78,REA,classic,fulfillment,free_shipping,1440.0
2,464801,2021-02-03,0,156.78,REA,classic,fulfillment,free_shipping,1440.0
3,464801,2021-02-04,0,156.78,REA,classic,fulfillment,free_shipping,1440.0
4,464801,2021-02-05,1,156.78,REA,classic,fulfillment,free_shipping,1440.0


## Dimensiones y tipos de datos — Dataset de entrenamiento
37M filas, 9 columnas. Período: 01/02/2021 al 31/03/2021.

In [42]:
print(archivo_parquet.shape)

(37660279, 9)


In [43]:
archivo_parquet.dtypes

sku                         int64
date                          str
sold_quantity               int64
current_price             float64
currency                      str
listing_type                  str
shipping_logistic_type        str
shipping_payment              str
minutes_active            float64
dtype: object

In [44]:
print(archivo_parquet['date'].min(),archivo_parquet['date'].max())

2021-02-01 2021-03-31


## Cobertura por SKU
Días de historial disponibles por producto. Mínimo 1 día, máximo 59.

In [45]:
dias_por_sku = archivo_parquet.groupby('sku').size()
print(dias_por_sku.min(), dias_por_sku.max())

1 59


## Distribución de ventas diarias
Más del 50% de los días no se registran ventas. Distribución de cola larga típica de e-commerce.

In [46]:
archivo_parquet['sold_quantity'].describe()

count    3.766028e+07
mean     9.900934e-01
std      9.989535e+00
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      1.000000e+00
max      6.951000e+03
Name: sold_quantity, dtype: float64

## Dataset de test — target_stock
551,472 SKUs a predecir. Stock mediano de 6 unidades. Máximo 32,710.

In [47]:
archivo_csv['sku'].count()

np.int64(551472)

In [49]:
archivo_csv['target_stock'].describe()

count    551472.000000
mean         18.069472
std         122.711151
min           1.000000
25%           3.000000
50%           6.000000
75%          13.000000
max       32710.000000
Name: target_stock, dtype: float64

## Join entre datasets
item_id del metadata coincide con sku del parquet. Los tres archivos se pueden unir por este campo.

In [61]:
df_merge = pd.merge(archivo_jl,archivo_parquet, left_on = 'item_id', right_on = 'sku')

In [ ]:
df_merge.shape
df_merge.head()

<bound method NDFrame.head of               item_domain_id  item_id  \
0               MLB-SNEAKERS   492155   
1               MLB-SNEAKERS   492155   
2               MLB-SNEAKERS   492155   
3               MLB-SNEAKERS   492155   
4               MLB-SNEAKERS   492155   
...                      ...      ...   
37657655  MLM-SURGICAL_MASKS   423179   
37657656  MLM-SURGICAL_MASKS   423179   
37657657  MLM-SURGICAL_MASKS   423179   
37657658  MLM-SURGICAL_MASKS   423179   
37657659  MLM-SURGICAL_MASKS   423179   

                                                 item_title site_id   sku_x  \
0           Tênis Masculino Olympikus Cyber Barato Promoçao     MLB       0   
1           Tênis Masculino Olympikus Cyber Barato Promoçao     MLB       0   
2           Tênis Masculino Olympikus Cyber Barato Promoçao     MLB       0   
3           Tênis Masculino Olympikus Cyber Barato Promoçao     MLB       0   
4           Tênis Masculino Olympikus Cyber Barato Promoçao     MLB       0   
...